In [7]:
import scanpy as sc
import anndata
import celltypist
from celltypist import train
import pandas as pd
import scipy.io
import scipy.sparse
import os

amp_dir = "/QRISdata/Q7458/Projects/ARBITRATE_singlecell/AMP Data"
out_dir = "/scratch/user/uqwsuwak/sc/AMP_ref"
os.makedirs(out_dir, exist_ok=True)

def load_from_export(export_dir):
    counts   = scipy.io.mmread(os.path.join(export_dir, "counts.mtx"))
    counts   = scipy.sparse.csr_matrix(counts).T   # cells x genes
    genes    = pd.read_csv(os.path.join(export_dir, "genes.csv"))
    barcodes = pd.read_csv(os.path.join(export_dir, "barcodes.csv"))
    meta     = pd.read_csv(os.path.join(export_dir, "metadata.csv"), index_col=0, encoding="latin-1")
    adata = anndata.AnnData(
        X   = counts,
        obs = meta,
        var = pd.DataFrame(index=genes["gene"].values),
    )
    adata.obs_names = barcodes["barcode"].values
    adata.var_names_make_unique()
    return adata

def train_model(export_dir, model_name):
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    adata = load_from_export(export_dir)
    print(f"  {adata.shape[0]} cells x {adata.shape[1]} genes")
    print("  Cell types:", adata.obs["clusters_names"].unique().tolist())

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    model = train(
        adata,
        labels="clusters_names",
        n_jobs=8,
        feature_selection=True,
        top_genes=300,
        check_expression=False,
    )

    model_path = os.path.join(out_dir, f"{model_name}.pkl")
    model.write(model_path)
    print(f"  Saved: {model_path}")
    print("  Cell types in model:")
    for ct in sorted(model.cell_types):
        print(f"    {ct}")

In [8]:
# ── Train separately ──────────────────────────────────────────────────────────
train_model(
    export_dir = os.path.join(amp_dir, "dunlapetal_finalCD4_export"),
    model_name = "dunlapetal_CD4_model"
)

train_model(
    export_dir = os.path.join(amp_dir, "dunlapetal_finalCD8_export"),
    model_name = "dunlapetal_CD8_model"
)

print("\nDone.")


Training: dunlapetal_CD4_model


/scratch/temp/23062393/ipykernel_934792/400409125.py:19: DtypeWarning: Columns (76,94,96) have mixed types. Specify dtype option on import or set low_memory=False.
  meta     = pd.read_csv(os.path.join(export_dir, "metadata.csv"), index_col=0, encoding="latin-1")


  35301 cells x 36601 genes
  Cell types: ['CD4_Naive', 'CD4_MT-high', 'CD4_CCR7+ memory', 'CD4_IL7R+CCL5+ memory', 'CD4_Proliferating', 'CD4_GNLY+', 'CD4_GIMAP+ naive/memory', 'CD4_GZMK+ memory', 'CD4_CD25-high Treg', 'CD4_CD25-low Treg', 'CD4_GZMA+CCL5+ memory', 'CD4_Tfh/Tph', 'CD4_Tph', 'CD4_ISG-high']


🍳 Preparing data before training
✂️ 11927 non-expressed genes are filtered out
🔬 Input data has 35301 cells and 24674 genes
⚖️ Scaling input data
🏋️ Training data using SGD logistic regression
🔎 Selecting features
🧬 2127 features are selected
🏋️ Starting the second round of training
🏋️ Training data using logistic regression
✅ Model training done!


  Saved: /scratch/user/uqwsuwak/sc/AMP_ref/dunlapetal_CD4_model.pkl
  Cell types in model:
    CD4_CCR7+ memory
    CD4_CD25-high Treg
    CD4_CD25-low Treg
    CD4_GIMAP+ naive/memory
    CD4_GNLY+
    CD4_GZMA+CCL5+ memory
    CD4_GZMK+ memory
    CD4_IL7R+CCL5+ memory
    CD4_ISG-high
    CD4_MT-high
    CD4_Naive
    CD4_Proliferating
    CD4_Tfh/Tph
    CD4_Tph

Training: dunlapetal_CD8_model


/scratch/temp/23062393/ipykernel_934792/400409125.py:19: DtypeWarning: Columns (76,94,96) have mixed types. Specify dtype option on import or set low_memory=False.
  meta     = pd.read_csv(os.path.join(export_dir, "metadata.csv"), index_col=0, encoding="latin-1")


  12547 cells x 36601 genes
  Cell types: ['CD8_Naive', 'CD8_GZMK/B+ memory', 'CD8_GZMB+ TEMRA', 'CD8_GZMK+ memory', 'CD8_ITGB1+ memory', 'CD8_Proliferating', 'CD8_Trm-like', 'CD8_MT-high', 'CD8_ISG-high']


🍳 Preparing data before training
✂️ 14545 non-expressed genes are filtered out
🔬 Input data has 12547 cells and 22056 genes
⚖️ Scaling input data
🏋️ Training data using SGD logistic regression
🔎 Selecting features
🧬 1709 features are selected
🏋️ Starting the second round of training
🏋️ Training data using logistic regression
✅ Model training done!


  Saved: /scratch/user/uqwsuwak/sc/AMP_ref/dunlapetal_CD8_model.pkl
  Cell types in model:
    CD8_GZMB+ TEMRA
    CD8_GZMK+ memory
    CD8_GZMK/B+ memory
    CD8_ISG-high
    CD8_ITGB1+ memory
    CD8_MT-high
    CD8_Naive
    CD8_Proliferating
    CD8_Trm-like

Done.
